# Chapter 11: Combining and Merging Data

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [92]:
import pandas as pd
import numpy as np

# Combining and Merging Data

Real-world data analysis rarely involves working with a single, perfectly organized dataset. Instead, you'll often need to combine information from multiple sources—customer data from one table, transaction history from another, and product information from a third. Pandas provides powerful tools to integrate these datasets efficiently and intuitively.

This chapter explores three fundamental operations for combining data: **concat()**, **merge()**, and **join()**. Understanding when and how to use each will transform your ability to work with complex, multi-source datasets.

---

## Understanding Relational Data Concepts

Before diving into the mechanics of combining datasets, let's establish some foundational concepts borrowed from relational database theory.

### Keys and Relationships

In relational data, a **key** is a column (or set of columns) that uniquely identifies a row. There are two main types:

- **Primary Key**: Uniquely identifies each row in a table
- **Foreign Key**: References a primary key in another table, establishing a relationship
- **Join Key**: The column(s) used to match rows between datasets

Consider this example:

In [93]:
import pandas as pd
import numpy as np

# Customers table with primary key 'customer_id'
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'city': ['New York', 'Los Angeles', 'Chicago', 'Houston']
})

# Orders table with foreign key 'customer_id'
orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 1, 3, 2],
    'amount': [150, 200, 75, 300, 125]
})

The `customer_id` in the orders table references the `customer_id` in the customers table, creating a relationship between them.

### Join Types

When combining datasets, you need to decide which rows to keep. This is where join types come in:

| Join Type | Behavior |
|-----------|----------|
| **Inner** | Keep only rows where keys match in both datasets |
| **Left** | Keep all rows from the left dataset; match from right where possible |
| **Right** | Keep all rows from the right dataset; match from left where possible |
| **Outer** | Keep all rows from both datasets |

---

## Concatenation with `concat()`

The `concat()` function is your go-to tool when you want to **stack** datasets together—combining them along rows or columns without worrying about matching keys.

### Understanding Vertical vs. Horizontal Stacking

Before diving into code, let's clarify what we mean by "stacking":

**Vertical Stacking (axis=0):** Adds more rows
```
DataFrame 1:        DataFrame 2:        Result (axis=0):
ID  Name            ID  Name            ID  Name
1   Alice           3   Carol           1   Alice
2   Bob             4   Diana           2   Bob
                                        3   Carol
                                        4   Diana
```

**Horizontal Stacking (axis=1):** Adds more columns
```
DataFrame 1:        DataFrame 2:        Result (axis=1):
ID  Name            Score               ID  Name  Score
1   Alice           95                  1   Alice 95
2   Bob             87                  2   Bob   87
```

### Real-World Scenario: Combining Quarterly Sales

Imagine you have quarterly sales data in separate files:
- `sales_q1.csv` — January to March
- `sales_q2.csv` — April to June
- `sales_q3.csv` — July to September
- `sales_q4.csv` — October to December

You need to combine all four quarters into one annual dataset for analysis. That's where `concat()` comes in.

### Stacking Rows (axis=0)

The most common use case is combining multiple DataFrames vertically:

In [94]:
import pandas as pd

# Create sample quarterly sales data
q1 = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar'],
    'Sales': [15000, 18000, 22000],
    'Region': ['North', 'South', 'East']
})

q2 = pd.DataFrame({
    'Month': ['Apr', 'May', 'Jun'],
    'Sales': [20000, 25000, 23000],
    'Region': ['North', 'South', 'East']
})

print("Q1 Data:")
print(q1)
print("\nQ2 Data:")
print(q2)

# Stack them vertically
all_sales = pd.concat([q1, q2], ignore_index=True)
print("\nCombined Sales (ignore_index=True):")
print(all_sales)

Q1 Data:
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East

Q2 Data:
  Month  Sales Region
0   Apr  20000  North
1   May  25000  South
2   Jun  23000   East

Combined Sales (ignore_index=True):
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
3   Apr  20000  North
4   May  25000  South
5   Jun  23000   East


**Output:**
```
Q1 Data:
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East

Q2 Data:
  Month  Sales Region
0   Apr  20000  North
1   May  25000  South
2   Jun  23000   East

Combined Sales (ignore_index=True):
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
3   Apr  20000  North
4   May  25000  South
5   Jun  23000   East
```

### The `ignore_index` Parameter Explained

The `ignore_index` parameter controls how indices are handled after concatenation.

**`ignore_index=False` (Default):** Preserves original indices

In [95]:
result_keep = pd.concat([q1, q2], ignore_index=False)
print(result_keep)

  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
0   Apr  20000  North
1   May  25000  South
2   Jun  23000   East


**Output:**
```
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
0   Apr  20000  North    # ← Index repeats!
1   May  25000  South
2   Jun  23000   East
```

Notice the index repeats (0, 1, 2, 0, 1, 2). This can cause confusion when accessing rows.

**`ignore_index=True`:** Creates a new sequential index

In [96]:
result_reset = pd.concat([q1, q2], ignore_index=True)
print(result_reset)

  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
3   Apr  20000  North
4   May  25000  South
5   Jun  23000   East


**Output:**
```
  Month  Sales Region
0   Jan  15000  North
1   Feb  18000  South
2   Mar  22000   East
3   Apr  20000  North    # ← Clean sequential index!
4   May  25000  South
5   Jun  23000   East
```

**When to use each:**
- Use `ignore_index=True` when you want a clean, sequential index (most common for combining datasets)
- Use `ignore_index=False` when you need to track which original DataFrame each row came from

### Adding Hierarchical Information with Keys

When concatenating multiple datasets, use the `keys` parameter to add a label identifying the source:

In [97]:
import pandas as pd

q1 = pd.DataFrame({
    'Month': ['Jan', 'Feb', 'Mar'],
    'Sales': [15000, 18000, 22000]
})

q2 = pd.DataFrame({
    'Month': ['Apr', 'May', 'Jun'],
    'Sales': [20000, 25000, 23000]
})

# Add keys to track source
combined = pd.concat([q1, q2], keys=['Q1', 'Q2'])
print(combined)

     Month  Sales
Q1 0   Jan  15000
   1   Feb  18000
   2   Mar  22000
Q2 0   Apr  20000
   1   May  25000
   2   Jun  23000


**Output:**
```
     Month  Sales
Q1 0   Jan  15000
   1   Feb  18000
   2   Mar  22000
Q2 0   Apr  20000
   1   May  25000
   2   Jun  23000
```

This creates a **MultiIndex** where you can easily filter by quarter:

In [98]:
# Filter Q1 sales only
print(combined.loc['Q1'])

  Month  Sales
0   Jan  15000
1   Feb  18000
2   Mar  22000


**Output:**
```
   Month  Sales
0   Jan  15000
1   Feb  18000
2   Mar  22000
```

### Handling Mismatched Columns

What happens if your DataFrames have different columns? Pandas handles this gracefully:

In [99]:
import pandas as pd

# DataFrames with different columns
df_a = pd.DataFrame({
    'Name': ['Alice', 'Bob'],
    'Score': [95, 87]
})

df_b = pd.DataFrame({
    'Name': ['Carol', 'Diana'],
    'Grade': ['A', 'B']  # Different column name!
})

# Concatenate with mismatched columns
combined = pd.concat([df_a, df_b], ignore_index=True)
print("Combined (default join='outer'):")
print(combined)

Combined (default join='outer'):
    Name  Score Grade
0  Alice   95.0   NaN
1    Bob   87.0   NaN
2  Carol    NaN     A
3  Diana    NaN     B


**Output:**
```
    Name  Score Grade
0  Alice   95.0   NaN
1    Bob   87.0   NaN
2  Carol    NaN     A
3  Diana    NaN     B
```

Notice the `NaN` (missing) values where columns don't exist. This is the default `join='outer'` behavior—it includes all columns from all DataFrames.

If you only want columns that exist in **all** DataFrames, use `join='inner'`:

In [100]:
combined_inner = pd.concat([df_a, df_b], join='inner', ignore_index=True)
print("Combined (join='inner'):")
print(combined_inner)

Combined (join='inner'):
    Name
0  Alice
1    Bob
2  Carol
3  Diana


**Output:**
```
Combined (join='inner'):
    Name
0  Alice
1    Bob
2  Carol
3  Diana
```

**When to use each:**
- `join='outer'` (default): Keep all columns, fill missing values with NaN
- `join='inner'`: Keep only columns that exist in all DataFrames

### Stacking Columns (axis=1)

Combine DataFrames side-by-side using `axis=1`. When using `axis=1`, DataFrames are joined **by index**, so make sure indices are aligned:

In [101]:
import pandas as pd

# Customer information
customer_info = pd.DataFrame({
    'ID': [1, 2, 3],
    'Name': ['Alice', 'Bob', 'Carol']
}, index=['ID_1', 'ID_2', 'ID_3'])

# Purchase history
purchase_history = pd.DataFrame({
    'Total_Spent': [500, 750, 320],
    'Last_Purchase': ['2024-01', '2024-02', '2024-01']
}, index=['ID_1', 'ID_2', 'ID_3'])

# Combine horizontally
combined = pd.concat([customer_info, purchase_history], axis=1)
print(combined)

      ID   Name  Total_Spent Last_Purchase
ID_1   1  Alice          500       2024-01
ID_2   2    Bob          750       2024-02
ID_3   3  Carol          320       2024-01


**Output:**
```
      ID     Name  Total_Spent Last_Purchase
ID_1   1    Alice           500      2024-01
ID_2   2      Bob           750      2024-02
ID_3   3    Carol           320      2024-01
```

If indices don't match, you'll get NaN values. The solution is to reset or align indices before concatenating:

In [102]:
import pandas as pd

df1 = pd.DataFrame({'Name': ['Alice', 'Bob', 'Carol']}, index=[1, 2, 3])
df2 = pd.DataFrame({'Score': [95, 87, 92]}, index=[1, 2, 4])  # Index 4 doesn't match!

# Default: outer join creates NaN values
combined = pd.concat([df1, df2], axis=1)
print(combined)

    Name  Score
1  Alice   95.0
2    Bob   87.0
3  Carol    NaN
4    NaN   92.0


**Output:**
```
    Name  Score
1  Alice   95.0
2    Bob   87.0
3  Carol    NaN     # ← No matching index 3 in df2
4    NaN   92.0    # ← No matching index 4 in df1
```

### Combining Many DataFrames Efficiently

When you have many files to combine, collect them in a list and call `concat()` once:

In [ ]:
import pandas as pd
import numpy as np
import os; os.makedirs('data', exist_ok=True)

# ----------------------------
# Create dummy quarterly files
# ----------------------------
np.random.seed(42)

for q in range(1, 5):
    df = pd.DataFrame({
        'order_id': range((q - 1) * 5 + 1, q * 5 + 1),
        'quarter': [f'Q{q}'] * 5,
        'product': np.random.choice(
            ['Laptop', 'Phone', 'Tablet', 'Monitor'],
            size=5
        ),
        'sales': np.random.randint(100, 5000, size=5)
    })

    filename = f'data/sales_q{q}.csv'
    df.to_csv(filename, index=False)
    print(f"Created {filename}")

# ----------------------------
# Read and combine all quarters
# ----------------------------
quarters = []

for i in range(1, 5):
    df = pd.read_csv(f'data/sales_q{i}.csv')
    quarters.append(df)

annual_sales = pd.concat(quarters, ignore_index=True)

print("\nAnnual Sales Data:")
print(annual_sales)

print("\nShape:", annual_sales.shape)
print("\nFirst 5 rows:")
print(annual_sales.head())

**Why not use `.append()` in a loop?**

The `.append()` method is **deprecated** and removed in recent pandas versions. Beyond that, calling it in a loop is slow because each call creates a new DataFrame copy. With `concat()`, you build a list once and combine it all at the end:

In [104]:
# ❌ SLOW and deprecated: Don't do this!
# result = pd.DataFrame()
# for i in range(1, 5):
#     df = pd.read_csv(f'sales_q{i}.csv')
#     result = result.append(df)  # Creates new copy each iteration

# ✅ Use this instead
result = pd.concat(quarters, ignore_index=True)

### Practical Example: Combining Quarterly Sales Data

Here's a complete real-world example:

In [105]:
import pandas as pd

# Create sample quarterly data
q1 = pd.DataFrame({
    'Date': ['2024-01-15', '2024-02-20', '2024-03-10'],
    'Product': ['Laptop', 'Mouse', 'Keyboard'],
    'Sales': [1200, 25, 45],
    'Region': ['North', 'South', 'East']
})

q2 = pd.DataFrame({
    'Date': ['2024-04-05', '2024-05-12', '2024-06-18'],
    'Product': ['Monitor', 'Laptop', 'Mouse'],
    'Sales': [350, 1150, 30],
    'Region': ['West', 'North', 'South']
})

q3 = pd.DataFrame({
    'Date': ['2024-07-22', '2024-08-14', '2024-09-30'],
    'Product': ['Keyboard', 'Monitor', 'Laptop'],
    'Sales': [50, 380, 1300],
    'Region': ['East', 'West', 'North']
})

# Combine all quarters
annual_sales = pd.concat([q1, q2, q3], ignore_index=True)

print("Annual Sales Data:")
print(annual_sales)

# Analysis
print("\nTotal Sales by Region:")
print(annual_sales.groupby('Region')['Sales'].sum())

print("\nAverage Sales by Product:")
print(annual_sales.groupby('Product')['Sales'].mean())

Annual Sales Data:
         Date   Product  Sales Region
0  2024-01-15    Laptop   1200  North
1  2024-02-20     Mouse     25  South
2  2024-03-10  Keyboard     45   East
3  2024-04-05   Monitor    350   West
4  2024-05-12    Laptop   1150  North
5  2024-06-18     Mouse     30  South
6  2024-07-22  Keyboard     50   East
7  2024-08-14   Monitor    380   West
8  2024-09-30    Laptop   1300  North

Total Sales by Region:
Region
East       95
North    3650
South      55
West      730
Name: Sales, dtype: int64

Average Sales by Product:
Product
Keyboard      47.500000
Laptop      1216.666667
Monitor      365.000000
Mouse         27.500000
Name: Sales, dtype: float64


**Output:**
```
Annual Sales Data:
         Date    Product  Sales Region
0  2024-01-15     Laptop   1200  North
1  2024-02-20      Mouse     25  South
2  2024-03-10   Keyboard     45   East
3  2024-04-05    Monitor    350   West
4  2024-05-12     Laptop   1150  North
5  2024-06-18      Mouse     30  South
6  2024-07-22   Keyboard     50   East
7  2024-08-14    Monitor    380   West
8  2024-09-30     Laptop   1300  North

Total Sales by Region:
Region
East       95
North    3650
South      55
West      730
Name: Sales, dtype: int64

Average Sales by Product:
Product
Keyboard      47.5
Laptop      1216.666667
Monitor      365.0
Mouse         27.5
Name: Sales, dtype: float64
```

---

## Merging with `merge()`

The `merge()` function performs database-style joins, combining datasets based on common columns or indices. This is essential when you need to match records across tables.

### Real-World Scenario: Merging Customer and Order Data

Suppose you have two datasets: one with customer details and another with their orders. You want to combine them to analyze total spending by customer.

In [106]:
import pandas as pd

# Create sample data
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'city': ['NYC', 'LA', 'Chicago', 'Boston']
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 2, 1, 5],
    'amount': [150, 200, 75, 300]
})

# Merge on customer ID
merged = pd.merge(customers, orders, on='customer_id', how='inner')
print(merged)

   customer_id   name city  order_id  amount
0            1  Alice  NYC       101     150
1            1  Alice  NYC       103      75
2            2    Bob   LA       102     200


**Output:**
```
   customer_id     name      city  order_id  amount
0            1    Alice       NYC       101     150
1            1    Alice       NYC       103      75
2            2      Bob        LA       102     200
```

**What happened?** We combined customer information with their orders using `customer_id` as the matching key. Customers 3 and 4 don't appear—they have no orders. Customer 5's order doesn't appear—they're not in the customer list.

### The Four Join Types

#### 1. Inner Join (Default)

**Keeps only rows where keys match in both DataFrames.**

In [107]:
import pandas as pd

left_df = pd.DataFrame({
    'ID': [1, 2, 3],
    'Name': ['Alice', 'Bob', 'Charlie']
})

right_df = pd.DataFrame({
    'ID': [2, 3, 4],
    'Score': [85, 90, 95]
})

inner = pd.merge(left_df, right_df, on='ID', how='inner')
print(inner)

   ID     Name  Score
0   2      Bob     85
1   3  Charlie     90


**Output:**
```
   ID     Name  Score
0   2      Bob     85
1   3  Charlie     90
```

Only IDs 2 and 3 appear in both DataFrames, so only they're included. Alice (ID 1) and the unknown person (ID 4) are dropped.

**Real-world use:** Finding customers who both made a purchase AND filled out a survey.

#### 2. Left Join

**Keeps all rows from the left DataFrame, plus matching data from the right.**

In [108]:
left = pd.merge(left_df, right_df, on='ID', how='left')
print(left)

   ID     Name  Score
0   1    Alice    NaN
1   2      Bob   85.0
2   3  Charlie   90.0


**Output:**
```
   ID     Name  Score
0   1    Alice    NaN
1   2      Bob   85.0
2   3  Charlie   90.0
```

All rows from the left DataFrame are kept. Alice's score is `NaN` because she's not in the right DataFrame.

**Real-world use:** Keeping all customers and adding their order data (if they have any).

#### 3. Right Join

**Keeps all rows from the right DataFrame, plus matching data from the left.**

In [109]:
right = pd.merge(left_df, right_df, on='ID', how='right')
print(right)

   ID     Name  Score
0   2      Bob     85
1   3  Charlie     90
2   4      NaN     95


**Output:**
```
   ID     Name  Score
0   2      Bob     85
1   3  Charlie     90
2   4      NaN     95
```

The person with ID 4 has no name (NaN) because they don't exist in the left DataFrame.

**Real-world use:** Keeping all survey responses and adding customer names (if available).

#### 4. Outer Join

**Keeps all rows from both DataFrames.**

In [110]:
outer = pd.merge(left_df, right_df, on='ID', how='outer')
print(outer)

   ID     Name  Score
0   1    Alice    NaN
1   2      Bob   85.0
2   3  Charlie   90.0
3   4      NaN   95.0


**Output:**
```
   ID     Name  Score
0   1    Alice    NaN
1   2      Bob   85.0
2   3  Charlie   90.0
3   4      NaN   95.0
```

All rows from both DataFrames are included. Unmatched records have `NaN` in the columns from the other DataFrame.

**Real-world use:** Creating a complete dataset of all customers and all survey respondents, even if they don't overlap.

### Visual Guide to Join Types

```
LEFT DataFrame:          RIGHT DataFrame:
ID  Name                 ID  Score
1   Alice                2   85
2   Bob                  3   90
3   Charlie              4   95

Join Type  |  Result IDs  |  Rows Included
-----------+--------------+------------------
INNER      |  2, 3        |  Only overlapping keys
LEFT       |  1, 2, 3     |  All from left
RIGHT      |  2, 3, 4     |  All from right
OUTER      |  1, 2, 3, 4  |  All from both
```

### Merging on Different Column Names

When the key columns have different names in each dataset, use `left_on` and `right_on`:

In [111]:
import pandas as pd

customers = pd.DataFrame({
    'cust_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie']
})

orders = pd.DataFrame({
    'customer_id': [1, 2, 4],
    'order_amount': [150, 200, 300]
})

# Use left_on and right_on
merged = pd.merge(customers, orders,
                  left_on='cust_id',
                  right_on='customer_id',
                  how='left')
print(merged)

   cust_id     name  customer_id  order_amount
0        1    Alice          1.0         150.0
1        2      Bob          2.0         200.0
2        3  Charlie          NaN           NaN


**Output:**
```
   cust_id     name  customer_id  order_amount
0        1    Alice          1.0         150.0
1        2      Bob          2.0         200.0
2        3  Charlie          NaN           NaN
```

Note that the merge created both `cust_id` and `customer_id` columns. Drop the duplicate if needed:

In [112]:
merged = merged.drop('customer_id', axis=1)

### Merging on Multiple Keys

When a single column isn't enough to uniquely identify records, merge on multiple columns:

In [113]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 1, 2, 2],
    'year': [2023, 2024, 2023, 2024],
    'name': ['Alice', 'Alice', 'Bob', 'Bob']
})

orders = pd.DataFrame({
    'customer_id': [1, 1, 2, 3],
    'year': [2023, 2024, 2024, 2023],
    'amount': [100, 150, 200, 250]
})

# Merge on multiple columns
merged = pd.merge(customers, orders,
                  on=['customer_id', 'year'],
                  how='left')
print(merged)

   customer_id  year   name  amount
0            1  2023  Alice   100.0
1            1  2024  Alice   150.0
2            2  2023    Bob     NaN
3            2  2024    Bob   200.0


**Output:**
```
   customer_id  year     name  amount
0            1  2023    Alice   100.0
1            1  2024    Alice   150.0
2            2  2023      Bob     NaN
3            2  2024      Bob   200.0
```

Records only match when BOTH `customer_id` AND `year` are the same. Bob's 2023 record has no order (NaN).

### Handling Duplicate Column Names

When merging DataFrames with overlapping column names (other than the join key), pandas automatically adds suffixes:

In [114]:
import pandas as pd

df_left = pd.DataFrame({
    'id': [1, 2],
    'value': [10, 20],
    'status': ['active', 'inactive']
})

df_right = pd.DataFrame({
    'id': [1, 2],
    'value': [100, 200],
    'notes': ['good', 'bad']
})

merged = pd.merge(df_left, df_right, on='id')
print(merged)

   id  value_x    status  value_y notes
0   1       10    active      100  good
1   2       20  inactive      200   bad


**Output:**
```
   id  value_x    status  value_y notes
0   1       10    active      100  good
1   2       20  inactive      200   bad
```

Customize suffixes with the `suffixes` parameter:

In [115]:
merged = pd.merge(df_left, df_right, on='id', suffixes=('_2023', '_2024'))

### Understanding Duplicate Keys and Many-to-Many Merges

What happens when the join key appears multiple times in one or both DataFrames?

In [116]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'name': ['Alice', 'Alice', 'Bob']
})

orders = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'order_id': [101, 102, 201]
})

merged = pd.merge(customers, orders, on='customer_id')
print(merged)

   customer_id   name  order_id
0            1  Alice       101
1            1  Alice       102
2            1  Alice       101
3            1  Alice       102
4            2    Bob       201


**Output:**
```
   customer_id     name  order_id
0            1    Alice       101
1            1    Alice       102
2            1    Alice       101
3            1    Alice       102
4            2      Bob       201
```

Customer 1 appears twice in both DataFrames, creating a **Cartesian product** (2 × 2 = 4 rows). This is a many-to-many merge.

**Warning:** Many-to-many merges can create unexpected row counts! Always verify your results.

### Using `validate` to Catch Merge Errors

The `validate` parameter helps you verify the relationship between keys and raises an error if the assumption is violated:

In [117]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie']
})

orders = pd.DataFrame({
    'customer_id': [1, 2, 2, 3],
    'order_id': [101, 102, 103, 104]
})

# One-to-many: each customer can have multiple orders
merged = pd.merge(customers, orders,
                  on='customer_id',
                  how='inner',
                  validate='1:m')
print(merged)

   customer_id     name  order_id
0            1    Alice       101
1            2      Bob       102
2            2      Bob       103
3            3  Charlie       104


Valid options: `'1:1'`, `'1:m'`, `'m:1'`, `'m:m'`

### Using `indicator=True` to Diagnose Merge Quality

The `indicator` parameter adds a `_merge` column showing where each row came from:

In [118]:
import pandas as pd

purchases = pd.DataFrame({
    'customer_id': [101, 102, 103, 104],
    'amount': [150, 200, 75, 300]
})

demographics = pd.DataFrame({
    'customer_id': [101, 102, 103, 105],
    'age': [28, 45, 32, 50]
})

merged = pd.merge(
    purchases,
    demographics,
    on='customer_id',
    how='outer',
    indicator=True
)

print("Merged with Indicator:")
print(merged)
print("\nMerge Quality Report:")
print(merged['_merge'].value_counts())

Merged with Indicator:
   customer_id  amount   age      _merge
0          101   150.0  28.0        both
1          102   200.0  45.0        both
2          103    75.0  32.0        both
3          104   300.0   NaN   left_only
4          105     NaN  50.0  right_only

Merge Quality Report:
_merge
both          3
left_only     1
right_only    1
Name: count, dtype: int64


**Output:**
```
   customer_id  amount   age     _merge
0          101   150.0  28.0       both
1          102   200.0  45.0       both
2          103    75.0  32.0       both
3          104   300.0   NaN  left_only
4          105     NaN  50.0 right_only

_merge
both          3
left_only     1
right_only    1
Name: count, dtype: int64
```

**Interpretation:**
- `both`: 3 customers matched perfectly
- `left_only`: 1 customer (104) in purchases but not in demographics
- `right_only`: 1 customer (105) in demographics but not in purchases

---

## Joining with `join()`

The `join()` method is a convenient shortcut for merging on indices. It's particularly useful when your key information is already stored as the index.

### Basic Index-Based Join

In [119]:
import pandas as pd

# Employee data indexed by employee_id
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'department': ['Sales', 'IT', 'HR']
}, index=pd.Index([101, 102, 103], name='employee_id'))

# Salary data indexed by employee_id
salaries = pd.DataFrame({
    'salary': [65000, 75000, 60000],
    'bonus': [5000, 8000, 4000]
}, index=pd.Index([101, 102, 103], name='employee_id'))

# Join on index (default: left join)
result = employees.join(salaries)
print(result)

                name department  salary  bonus
employee_id                                   
101            Alice      Sales   65000   5000
102              Bob         IT   75000   8000
103          Charlie         HR   60000   4000


**Output:**
```
               name department  salary  bonus
employee_id
101           Alice      Sales   65000   5000
102             Bob         IT   75000   8000
103         Charlie         HR   60000   4000
```

### Specifying Join Type

In [120]:
# Left join (default)
result_left = employees.join(salaries, how='left')

# Inner join
result_inner = employees.join(salaries, how='inner')

# Outer join
result_outer = employees.join(salaries, how='outer')

### Joining Multiple DataFrames at Once

In [121]:
performance = pd.DataFrame({
    'rating': [4.5, 3.8, 4.2]
}, index=pd.Index([101, 102, 103], name='employee_id'))

# Join multiple DataFrames at once
result = employees.join([salaries, performance])
print(result)

                name department  salary  bonus  rating
employee_id                                           
101            Alice      Sales   65000   5000     4.5
102              Bob         IT   75000   8000     3.8
103          Charlie         HR   60000   4000     4.2


**Output:**
```
               name department  salary  bonus  rating
employee_id
101           Alice      Sales   65000   5000     4.5
102             Bob         IT   75000   8000     3.8
103         Charlie         HR   60000   4000     4.2
```

### Joining on a Column to an Index

When the join key is a column in one dataset and the index in another:

In [122]:
import pandas as pd

# Employee data with department_id column
employees = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Charlie'],
    'department_id': [1, 2, 1]
})

# Department data indexed by department_id
departments = pd.DataFrame({
    'dept_name': ['Sales', 'IT', 'HR']
}, index=pd.Index([1, 2, 3], name='department_id'))

# Join on column to index
result = employees.join(departments, on='department_id')
print(result)

      name  department_id dept_name
0    Alice              1     Sales
1      Bob              2        IT
2  Charlie              1     Sales


**Output:**
```
      name  department_id dept_name
0    Alice              1     Sales
1      Bob              2        IT
2  Charlie              1     Sales
```

---

## Choosing the Right Tool: `concat()` vs. `merge()` vs. `join()`

| Operation | Best For | Key Feature |
|-----------|----------|-------------|
| **concat()** | Stacking datasets vertically or horizontally | Simple row/column combination |
| **merge()** | Database-style joins on column values | Flexible key matching, multiple join types |
| **join()** | Index-based combination | Convenient shorthand for index-aligned data |

**Quick Decision Tree:**
```
Are you stacking multiple datasets vertically or horizontally?
├─ YES → Use pd.concat()
└─ NO → Do you need to match rows by column values?
    ├─ YES → Use pd.merge() (most flexible)
    └─ Joining on indices? → Use df.join() (convenient shorthand)
```

The same result can often be achieved multiple ways:

In [123]:
import pandas as pd

df1 = pd.DataFrame({'key': [1, 2], 'A': [10, 20]})
df2 = pd.DataFrame({'key': [1, 2], 'B': [100, 200]})

# concat: requires setting index first
result1 = pd.concat([df1.set_index('key'), df2.set_index('key')], axis=1)

# merge: column-based (most explicit)
result2 = pd.merge(df1, df2, on='key')

# join: index-based
result3 = df1.set_index('key').join(df2.set_index('key'))

All three produce the same result, but `merge()` is most explicit about the relationship. **Recommendation for beginners:** Use `merge()` with explicit column names—it's clearer and less error-prone.

---

## Common Pitfalls and Solutions

### Pitfall 1: Silent Data Loss with Inner Joins

In [124]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve']
})

orders = pd.DataFrame({
    'customer_id': [1, 2, 2, 4],
    'order_id': [101, 102, 103, 104]
})

# Problem: Inner join silently drops unmatched rows
inner = pd.merge(customers, orders, on='customer_id', how='inner')
print(f"Original customers: {len(customers)}")
print(f"After inner join: {len(inner)}")  # Fewer rows!

# Solution: Use left join first to see what's being dropped
left = pd.merge(customers, orders, on='customer_id', how='left')
unmatched = left[left['order_id'].isna()]
print(f"\nUnmatched customers: {len(unmatched)}")
print(unmatched)

Original customers: 5
After inner join: 4

Unmatched customers: 2
   customer_id     name  order_id
3            3  Charlie       NaN
5            5      Eve       NaN


### Pitfall 2: Cartesian Product from Duplicate Keys

In [125]:
import pandas as pd

# Problem: Duplicate keys create unexpected row multiplication
customers = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'name': ['Alice', 'Alice', 'Bob']
})

orders = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'order_id': [101, 102, 201]
})

result = customers.merge(orders, on='customer_id')
print(f"Expected ~3 rows, got {len(result)} rows!")
print("This is a many-to-many join (2 × 2 = 4 rows for customer_id=1)")

# Solution: Check for duplicates before merging
print(f"\nDuplicate customers: {customers['customer_id'].duplicated().sum()}")
customers_clean = customers.drop_duplicates(subset=['customer_id'], keep='first')
result_clean = customers_clean.merge(orders, on='customer_id')
print(f"After deduplication: {len(result_clean)} rows")

Expected ~3 rows, got 5 rows!
This is a many-to-many join (2 × 2 = 4 rows for customer_id=1)

Duplicate customers: 1
After deduplication: 3 rows


### Pitfall 3: Data Type Mismatches in Join Keys

In [126]:
import pandas as pd

# Problem: Different data types prevent matching
df_a = pd.DataFrame({'key': [1, 2, 3]})       # Integer
df_b = pd.DataFrame({'key': ['1', '2', '3']}) # String

result = pd.concat([df_a, df_b], ignore_index=True)
print(f"Rows after concat: {len(result)}")  # 0 rows! No match!

# Solution: Convert to same type
df_b['key'] = df_b['key'].astype(int)
result = pd.concat([df_a, df_b], ignore_index=True)
print(f"Rows after fixing types: {len(result)}")  # 6 rows

Rows after concat: 6
Rows after fixing types: 6


**Output:**
```
Rows after merge: 0
Rows after fixing types: 3
```

Pandas merge requires matching data types on the join keys. When types don't match (int vs string), no rows merge even if the values look identical.

### Pitfall 4: Case Sensitivity and Whitespace in String Keys

In [127]:
import pandas as pd

# Problem: String keys are case-sensitive and whitespace-sensitive
df1 = pd.DataFrame({'region': ['North', 'South', 'East'], 'sales': [1000, 1500, 1200]})
df2 = pd.DataFrame({'region': ['north', ' South', 'EAST'], 'target': [1200, 1400, 1300]})

result = pd.merge(df1, df2, on='region')
print(f"Merge result: {len(result)} rows")  # 0 rows!

# Solution: Standardize before merging
df1['region'] = df1['region'].str.strip().str.lower()
df2['region'] = df2['region'].str.strip().str.lower()
result = pd.merge(df1, df2, on='region')
print(f"After standardizing: {len(result)} rows")
print(result)

Merge result: 0 rows
After standardizing: 3 rows
  region  sales  target
0  north   1000    1200
1  south   1500    1400
2   east   1200    1300


---

## Practical Workflow: Combining Multiple Data Sources

Let's work through a realistic scenario combining all three techniques:

In [128]:
import pandas as pd

# Load data from different sources
transactions = pd.DataFrame({
    'transaction_id': [1, 2, 3, 4, 5],
    'customer_id': [101, 102, 101, 103, 102],
    'amount': [250, 150, 300, 200, 175]
})

customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'region': ['North', 'South', 'East', 'West']
})

# Step 1: Merge transactions with customer info
df_combined = pd.merge(transactions, customers, on='customer_id', how='left')

# Step 2: Add regional summary
regional_stats = df_combined.groupby('region')['amount'].agg(['sum', 'mean']).reset_index()
regional_stats.columns = ['region', 'total_sales', 'avg_transaction']

# Step 3: Join regional stats back
result = df_combined.merge(regional_stats, on='region', how='left')
print(result)

   transaction_id  customer_id  amount     name region  total_sales  \
0               1          101     250    Alice  North          550   
1               2          102     150      Bob  South          325   
2               3          101     300    Alice  North          550   
3               4          103     200  Charlie   East          200   
4               5          102     175      Bob  South          325   

   avg_transaction  
0            275.0  
1            162.5  
2            275.0  
3            200.0  
4            162.5  


**Output:**
```
   transaction_id  customer_id  amount     name region  total_sales  avg_transaction
0               1          101     250    Alice  North          550           275.0
1               2          102     150      Bob  South          325           162.5
2               3          101     300    Alice  North          550           275.0
3               4          103     200  Charlie   East          200           200.0
4               5          102     175      Bob  South          325           162.5
```

---

## Checking Your Merge Results

After merging, always verify the results:

In [129]:
import pandas as pd

# After performing a merge, validate the output
def check_merge(left_df, right_df, merged_df, key_col):
    print("=== MERGE VALIDATION ===")
    print(f"Left rows:   {len(left_df)}")
    print(f"Right rows:  {len(right_df)}")
    print(f"Result rows: {len(merged_df)}")
    print(f"\nNaN values:\n{merged_df.isnull().sum()}")
    print(f"\nKey value counts:\n{merged_df[key_col].value_counts().head()}")

# Example usage
customers = pd.DataFrame({'customer_id': [1, 2, 3], 'name': ['Alice', 'Bob', 'Charlie']})
orders = pd.DataFrame({'customer_id': [1, 2, 2], 'amount': [100, 200, 150]})
merged = pd.merge(customers, orders, on='customer_id', how='left')
check_merge(customers, orders, merged, 'customer_id')

=== MERGE VALIDATION ===
Left rows:   3
Right rows:  3
Result rows: 4

NaN values:
customer_id    0
name           0
amount         1
dtype: int64

Key value counts:
customer_id
2    2
1    1
3    1
Name: count, dtype: int64


---

## Performance Considerations

When combining large datasets, keep these tips in mind:

1. **Use `concat()` for simple stacking** — it's faster than repeated appends.
2. **Filter before joining** — reduce memory usage by filtering each dataset before combining.
3. **Use indexed joins for large datasets** — index-based joins are significantly faster than column-based merges on large data.

In [130]:
import pandas as pd

# For large merges, indexed joins are faster
large_left = pd.DataFrame({'customer_id': range(100000), 'value': range(100000)})
large_right = pd.DataFrame({'customer_id': range(50000, 150000), 'data': range(100000)})

# Faster approach: Use indexed join
large_left_indexed = large_left.set_index('customer_id')
large_right_indexed = large_right.set_index('customer_id')
result_fast = large_left_indexed.join(large_right_indexed, how='inner')

4. **Use `validate` to catch errors early** — prevents silent many-to-many explosions:

In [131]:
# Raises an error if duplicates exist
merged = pd.merge(large_left, large_right, on='customer_id', validate='1:1')

---

## Summary

| Operation | Best For | Key Feature |
|-----------|----------|-------------|
| **concat()** | Stacking datasets | Simple row/column combination |
| **merge()** | Database-style joins | Flexible key matching, multiple join types |
| **join()** | Index-based combination | Convenient for index-aligned data |

**Key takeaways:**

- Use `pd.concat()` to stack DataFrames — it's flexible and efficient
- Use `ignore_index=True` for clean sequential indices when combining rows
- Use `axis=0` for vertical stacking (more rows), `axis=1` for horizontal stacking (more columns)
- Handle mismatched columns with `join='outer'` (default) or `join='inner'`
- Use `keys` parameter to track which DataFrame each row came from
- Collect DataFrames in a list, then `concat()` once — never use `.append()` in loops
- Use `pd.merge()` for database-style joins on column values
- Always specify `how` and `on` parameters explicitly for clarity
- Use `indicator=True` to diagnose merge quality
- Validate merge results by checking row counts and NaN values
- Watch for silent data loss (inner joins), Cartesian products (duplicate keys), and type mismatches

---

## Practice Exercises

1. **Create two DataFrames** with customer and product data, then merge them using all four join types and observe the differences.
2. **Concatenate** three monthly sales reports into one annual dataset, adding a `month` column to each before combining.
3. **Merge on multiple columns** to match records by both `customer_id` and `year`.
4. **Use `indicator=True`** to identify unmatched records after an outer join, then investigate why they didn't match.
5. **Identify and fix** a merge that produces 0 rows due to a data type mismatch between join keys.
6. **Build a complete analysis pipeline**: start with separate transactions, customers, and products tables; merge them together; then compute total revenue by product category.

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Stacking Sales DataFrames with concat()

You have two quarterly sales DataFrames with identical columns. Use pd.concat() to stack them vertically into a single DataFrame. Make sure to reset the index so it runs continuously from 0.

In [ ]:
import pandas as pd

q1_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'units_sold': [120, 85, 200],
    'revenue': [2400, 1700, 4000]
})

q2_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'units_sold': [150, 90, 175],
    'revenue': [3000, 1800, 3500]
})

# TODO: Concatenate q1_sales and q2_sales vertically
combined = None

# TODO: Reset the index so it runs from 0 to 5 (drop the old index)
combined = None

print(combined)
print("\nShape:", combined.shape if combined is not None else 'Fill in the exercise above')

### Exercise 2: Merging Customer and Order Data

You have a customers DataFrame and an orders DataFrame linked by a customer_id column. Perform an inner merge to combine them, then perform a left merge to keep all customers even if they have no orders. Compare the row counts of both results.

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'city': ['New York', 'Boston', 'Chicago', 'Boston', 'New York']
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 2, 1, 3],
    'amount': [250.0, 89.99, 430.0, 120.50]
})

# TODO: Perform an inner merge on 'customer_id'
inner_merged = None

# TODO: Perform a left merge on 'customer_id' to keep all customers
left_merged = None

print("Inner merge result:")
print(inner_merged)
print("\nLeft merge result:")
print(left_merged)

# TODO: Print the number of rows in each merged DataFrame
print("\nInner merge rows:", None)
print("Left merge rows:", None)

### Exercise 3: Joining DataFrames on Their Index with join()

You have two DataFrames indexed by employee ID — one with employee names and departments, and one with salary information. Use the DataFrame .join() method to combine them on their shared index. Then identify any employees missing salary data.

In [ ]:
import pandas as pd

employee_info = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David'],
    'department': ['Engineering', 'Marketing', 'Engineering', 'HR']
}, index=[101, 102, 103, 104])

employee_salary = pd.DataFrame({
    'salary': [95000, 72000, 88000],
    'bonus': [5000, 3000, 4500]
}, index=[101, 102, 103])

# TODO: Use .join() to combine employee_info with employee_salary
# Keep all employees from employee_info (left join behavior)
joined = None

print(joined)

# TODO: Find and print the names of employees with missing salary data
missing_salary = None
print("\nEmployees with no salary data:")
print(missing_salary)

### Exercise 4: Diagnosing and Fixing a Merge Problem

A merge between a products DataFrame and a reviews DataFrame is producing an unexpectedly large result due to duplicate keys — a common pitfall called a many-to-many join explosion. Investigate the issue by checking for duplicate product_ids in each DataFrame, then fix it by aggregating the reviews before merging.

In [ ]:
import pandas as pd

products = pd.DataFrame({
    'product_id': [1, 2, 3],
    'product_name': ['Laptop', 'Mouse', 'Keyboard'],
    'price': [999.99, 29.99, 79.99]
})

reviews = pd.DataFrame({
    'product_id': [1, 1, 2, 2, 2, 3],
    'reviewer': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank'],
    'rating': [5, 4, 3, 5, 4, 2]
})

# TODO: Merge products and reviews directly and print the shape
# Notice how many rows are produced
raw_merge = None
print("Raw merge shape:", None)
print(raw_merge)

# TODO: Check for duplicate product_ids in the reviews DataFrame
print("\nDuplicate product_ids in reviews:")
print(None)

# TODO: Fix the issue by aggregating reviews first:
# Calculate the mean rating and review count per product_id
reviews_summary = None

# TODO: Now merge products with the aggregated reviews_summary
clean_merge = None

print("\nClean merge result:")
print(clean_merge)

---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Stacking Sales DataFrames with concat()

In [ ]:
import pandas as pd
import numpy as np

q1_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'units_sold': [120, 85, 200],
    'revenue': [2400, 1700, 4000]
})

q2_sales = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C'],
    'units_sold': [150, 90, 175],
    'revenue': [3000, 1800, 3500]
})

# Concatenate q1_sales and q2_sales vertically
combined = pd.concat([q1_sales, q2_sales])

# Reset the index so it runs from 0 to 5 (drop the old index)
combined = combined.reset_index(drop=True)

print(combined)
print("\nShape:", combined.shape)

### Solution 2: Merging Customer and Order Data

In [ ]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Alice', 'Bob', 'Carol', 'David', 'Eve'],
    'city': ['New York', 'Boston', 'Chicago', 'Boston', 'New York']
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 2, 1, 3],
    'amount': [250.0, 89.99, 430.0, 120.50]
})

# Perform an inner merge on 'customer_id'
inner_merged = pd.merge(customers, orders, on='customer_id', how='inner')

# Perform a left merge on 'customer_id' to keep all customers
left_merged = pd.merge(customers, orders, on='customer_id', how='left')

print("Inner merge result:")
print(inner_merged)
print("\nLeft merge result:")
print(left_merged)

# Print the number of rows in each merged DataFrame
print("\nInner merge rows:", len(inner_merged))
print("Left merge rows:", len(left_merged))

### Solution 3: Joining DataFrames on Their Index with join()

In [ ]:
import pandas as pd
import numpy as np

employee_info = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol', 'David'],
    'department': ['Engineering', 'Marketing', 'Engineering', 'HR']
}, index=[101, 102, 103, 104])

employee_salary = pd.DataFrame({
    'salary': [95000, 72000, 88000],
    'bonus': [5000, 3000, 4500]
}, index=[101, 102, 103])

# Use .join() to combine employee_info with employee_salary
# Keep all employees from employee_info (left join behavior)
joined = employee_info.join(employee_salary, how='left')

print(joined)

# Find and print the names of employees with missing salary data
missing_salary = joined[joined['salary'].isna()]['name']
print("\nEmployees with no salary data:")
print(missing_salary)

### Solution 4: Diagnosing and Fixing a Merge Problem

In [ ]:
import pandas as pd
import numpy as np

products = pd.DataFrame({
    'product_id': [1, 2, 3],
    'product_name': ['Laptop', 'Mouse', 'Keyboard'],
    'price': [999.99, 29.99, 79.99]
})

reviews = pd.DataFrame({
    'product_id': [1, 1, 2, 2, 2, 3],
    'reviewer': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank'],
    'rating': [5, 4, 3, 5, 4, 2]
})

# Merge products and reviews directly and print the shape
# Notice how many rows are produced
raw_merge = pd.merge(products, reviews, on='product_id')
print("Raw merge shape:", raw_merge.shape)
print(raw_merge)

# Check for duplicate product_ids in the reviews DataFrame
print("\nDuplicate product_ids in reviews:")
print(reviews['product_id'].value_counts())

# Fix the issue by aggregating reviews first:
# Calculate the mean rating and review count per product_id
reviews_summary = reviews.groupby('product_id').agg(
    avg_rating=('rating', 'mean'),
    review_count=('rating', 'count')
).reset_index()

# Now merge products with the aggregated reviews_summary
clean_merge = pd.merge(products, reviews_summary, on='product_id', how='left')

print("\nClean merge result:")
print(clean_merge)